In [7]:
def berlekamp_massey_custom(syndromes, F):
    """
    Implémentation manuelle de l'algorithme de Berlekamp-Massey.
    """
    R.<x> = PolynomialRing(F)
    L = 0  # Nombre d'erreurs estimées
    Lambda = R(1)
    B = R(1)
    b = 1  # Divergence précédente
    m = 1  # Décalage
    
    for n in range(len(syndromes)):
        # Calcul de la divergence (discrepancy)
        d = syndromes[n]
        for i in range(1, L + 1):
            if i < len(Lambda.list()):
                d += Lambda.list()[i] * syndromes[n - i]
        
        if d == 0:
            m += 1
        else:
            old_Lambda = Lambda
            Lambda = Lambda - (d / b) * x^m * B
            if 2 * L <= n:
                L = n + 1 - L
                B = old_Lambda
                b = d
                m = 1
            else:
                m += 1
    return Lambda

In [9]:
# Paramètres du code RS [15, 9, 7] sur GF(16)
q = 16
F.<a> = GF(q)
n, k = 15, 9
t = (n - k) // 2  # Capacité : 3 erreurs

# Construction du code RS
C_RS = codes.ReedSolomonCode(F, n, k)

# 1. Codage
msg = vector(F, [F.random_element() for _ in range(k)])
c = C_RS.encode(msg)

# 2. Simulation d'erreurs (t=3)
y = copy(c)
y[1] += a; y[5] += 1; y[10] += a^2

# 3. Décodage par Berlekamp-Massey (KeyEquationSyndrome)
# Utilisation du nom correct du décodeur selon votre message d'erreur
try:
    # On peut soit utiliser "KeyEquationSyndrome", soit laisser Sage choisir
    D_RS = C_RS.decoder("KeyEquationSyndrome")
    res_rs = D_RS.decode_to_code(y)
    print("--- Reed-Solomon ---")
    print(f"Succès du décodage : {res_rs == c}")
    print(f"Mot corrigé : {res_rs}")
except Exception as e:
    print(f"Erreur : {e}")

--- Reed-Solomon ---
Succès du décodage : True
Mot corrigé : (a^3 + a, 0, 1, 0, 0, a^3 + a^2 + a + 1, a^3, a^3 + a + 1, a^3 + a^2, a + 1, a^3 + a^2 + 1, a^3 + a + 1, a^3, a^3, a^2 + a)


In [10]:
# Paramètres GRS [10, 6, 5] sur GF(11)
q_grs = 11
F_grs = GF(q_grs)
n_grs, k_grs = 10, 6

# Points d'évaluation
x_pts = [F_grs(i) for i in range(n_grs)]
C_GRS = codes.GeneralizedReedSolomonCode(x_pts, k_grs)

# 1. Codage
msg_grs = vector(F_grs, [F_grs.random_element() for _ in range(k_grs)])
c_grs = C_GRS.encode(msg_grs)

# 2. Erreurs (t=2)
y_grs = copy(c_grs)
y_grs[0] += F_grs(1); y_grs[2] += F_grs(4)

# 3. Décodage
try:
    # 'Gao' est souvent le décodeur le plus robuste pour les GRS dans Sage
    D_GRS = C_GRS.decoder("Gao") 
    res_grs = D_GRS.decode_to_code(y_grs)
    print("\n--- Reed-Solomon Généralisé ---")
    print(f"Décodage Gao réussi : {res_grs == c_grs}")
except Exception as e:
    print(f"Échec GRS : {e}")


--- Reed-Solomon Généralisé ---
Décodage Gao réussi : True
